# XÂY DỰNG MÔ HÌNH PHÂN CỤM K-MEANS TRÊN TẬP DỮ LIỆU MUA SẮM TẠI SIÊU THỊ

## 1. Mục tiêu

Mục tiêu của bài thực hành là xây dựng mô hình phân cụm K-Means trên tập dữ liệu OnlineRetail nhằm phân nhóm khách hàng dựa trên hành vi mua sắm. Thông qua mô hình, ta có thể nhận diện các nhóm khách hàng khác nhau (như khách mua thường xuyên, khách mua giá trị cao, khách ít mua…), từ đó hỗ trợ doanh nghiệp đưa ra chiến lược marketing, chăm sóc khách hàng và định giá phù hợp.

## 2. Import thư viện và nạp dữ liệu

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# Đọc dữ liệu
df = pd.read_csv("OnlineRetail.csv", encoding='latin1')

print("5 dòng đầu tiên của dữ liệu:")
print(df.head())

print("Thông tin dữ liệu:")
print(df.info())

5 dòng đầu tiên của dữ liệu:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice  CustomerID         Country  
0  01-12-2010 08:26       2.55     17850.0  United Kingdom  
1  01-12-2010 08:26       3.39     17850.0  United Kingdom  
2  01-12-2010 08:26       2.75     17850.0  United Kingdom  
3  01-12-2010 08:26       3.39     17850.0  United Kingdom  
4  01-12-2010 08:26       3.39     17850.0  United Kingdom  
Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  


**Nhận xét dữ liệu ban đầu:**

- Dữ liệu gồm 541.909 dòng và 8 cột, chứa thông tin giao dịch của một cửa hàng bán lẻ trực tuyến trong năm 2010–2011.

- Một số cột dạng chuỗi (InvoiceNo, StockCode, Description, Country), trong khi Quantity, UnitPrice là số, và CustomerID thuộc kiểu float nhưng thực chất là mã khách hàng.

- CustomerID có nhiều giá trị bị thiếu (khoảng 406.829 giá trị hợp lệ trên 541.909 dòng), điều này cần xử lý vì phân cụm khách hàng không thể thực hiện nếu không có mã khách hàng.

- Cột InvoiceDate đang ở dạng object, cần chuyển sang dạng datetime để phục vụ tính toán Recency.

- Các cột Quantity và UnitPrice có thể chứa giá trị âm hoặc bất thường (dataset OnlineRetail gốc thường có), cần kiểm tra và loại bỏ vì chúng ảnh hưởng rất mạnh đến RFM và phân cụm.

In [3]:
# Loại bỏ các dòng có InvoiceNo rỗng và Quantity/UnitPrice âm
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

df.head().style.background_gradient(cmap=sns.cubehelix_palette(as_cmap=True))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.550000,17850.000000,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.390000,17850.000000,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.750000,17850.000000,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.390000,17850.000000,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.390000,17850.000000,United Kingdom


Ở bước này, các giao dịch không có CustomerID và những dòng có Quantity hoặc UnitPrice âm đã được loại bỏ. Đây đều là những dữ liệu không hợp lệ hoặc không dùng được cho phân cụm. Sau khi lọc, dữ liệu trở nên “sạch” và chính xác hơn, phù hợp để tiếp tục phân tích và xây dựng mô hình.

## 3. EDA & Xử lý dữ liệu

### Tạo đặc trưng cho phân cụm

Ta dùng 3 đặc trưng phổ biến trong phân tích RFM:

- Recency: Số ngày kể từ lần mua gần nhất

- Frequency: Số lần mua hàng

- Monetary: Tổng số tiền đã mua

In [5]:
# Chuyển InvoiceDate sang datetime đúng định dạng dd-mm-yyyy
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], dayfirst=True, errors='coerce')

# Kiểm tra lỗi parse
print("Số dòng không đọc được ngày:", df['InvoiceDate'].isna().sum())

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Quantity': 'sum',
    'UnitPrice': 'mean'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'TotalQuantity', 'AvgPrice']

# Monetary = Quantity × AvgPrice
rfm['Monetary'] = rfm['TotalQuantity'] * rfm['AvgPrice']

rfm.head()

Số dòng không đọc được ngày: 0


,CustomerID,Recency,Frequency,TotalQuantity,AvgPrice,Monetary
0,12346.0,326,1,74215,1.040000,77183.600000
1,12347.0,2,7,2458,2.644011,6498.979011
2,12348.0,75,4,2341,5.764839,13495.487419
3,12349.0,19,1,631,8.289041,5230.384932
4,12350.0,310,1,197,3.841176,756.711765


Ở bước này, dữ liệu đã được chuyển về dạng ngày tháng đúng chuẩn, sau đó tính ra các chỉ số RFM cho từng khách hàng. Recency thể hiện số ngày kể từ lần mua cuối, Frequency là số lần mua, còn Monetary được tính từ số lượng và giá trung bình. Kết quả RFM nhìn khá rõ ràng, không có lỗi ngày tháng, và mỗi khách hàng đã có bộ thông tin đầy đủ để đưa vào mô hình phân cụm.

### Boxplot kiểm tra outlier

In [9]:
fig = px.box(rfm[['Recency','Frequency','Monetary']], log_y=True,
             title="Phân bố RFM – log scale")
fig.show()

Biểu đồ boxplot theo log-scale cho thấy cả ba biến Recency, Frequency và Monetary đều có outliers, đặc biệt là Monetary với nhiều giá trị rất lớn. Đây là hiện tượng bình thường trong dữ liệu bán lẻ vì một số khách hàng mua số lượng cao hoặc đơn hàng giá trị lớn. Việc nhận diện outlier giúp tránh để các giá trị “quá đỉnh” làm méo kết quả khi phân cụm.

### Xử lý Outliers (IQR Method)

In [10]:
def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

for feature in ['Recency','Frequency','Monetary']:
    rfm = remove_outliers(rfm, feature)

In [11]:
fig = px.box(rfm[['Recency','Frequency','Monetary']],
             log_y=True,
             title="Phân bố RFM sau khi xử lý outliers (log scale)")
fig.show()

Sau khi loại bỏ outliers bằng phương pháp IQR, boxplot cho thấy phân bố RFM đã gọn hơn và không còn các giá trị quá cực đoan. Dữ liệu trở nên ổn định hơn, phù hợp để đưa vào bước chuẩn hóa và phân cụm K-Means.

### Chuẩn hóa dữ liệu

In [12]:
scaler = MinMaxScaler()
X = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

Ở bước này giúp đưa các biến Recency, Frequency và Monetary về cùng một thang đo. Vì các biến này có mức độ chênh lệch rất lớn (ví dụ Monetary có thể lên đến hàng nghìn trong khi Frequency chỉ vài lần), nếu không chuẩn hóa thì mô hình K-Means sẽ bị “lệch” và chỉ quan tâm đến biến có giá trị lớn nhất. Việc chuẩn hóa đảm bảo mọi biến đều được mô hình xem trọng như nhau và giúp phân cụm chính xác hơn.

## 4. Tìm số cluster tối ưu (Elbow Method)

In [14]:
sse = []
for i in range(1, 10):
    kmeans = KMeans(n_clusters=i, max_iter=300)
    kmeans.fit(X)
    sse.append(kmeans.inertia_)

fig = px.line(y=sse, template='seaborn',
              title='Elbow Method for K Selection')
fig.update_layout(width=800, height=500,
                  xaxis_title="Clusters", yaxis_title="SSE")
fig.show()

**Nhận xét biểu đồ Elbow:**

Dựa trên biểu đồ Elbow, ta thấy SSE giảm mạnh từ k = 1 đến k = 3, sau đó đường cong bắt đầu phẳng dần. Điểm gấp khúc (elbow) nằm khoảng k = 3, nghĩa là mô hình đạt sự cân bằng tốt giữa số cụm và độ chính xác khi phân nhóm. Vì vậy, k = 3 được xem là số cụm tối ưu cho bài toán phân cụm khách hàng dựa trên RFM.

## 5. Xây dựng mô hình K-Means với k = 3

In [15]:
kmeans = KMeans(
    n_clusters=3,
    init='k-means++',
    max_iter=300,
    n_init=10,
    random_state=42
)

clusters = kmeans.fit_predict(X)
rfm['Cluster'] = clusters

## 6. Trình bày kết quả trực quan

### Scatter plot R – F theo cụm

In [27]:
# Chuyển Cluster thành dạng chuỗi để plot màu rõ ràng
rfm['Cluster_str'] = rfm['Cluster'].astype(str)

fig = px.scatter(
    rfm, x='Recency', y='Frequency',
    color='Cluster_str',
    opacity=0.55,                    # giúp giảm chồng điểm
    title="K-Means Clustering (R–F) – Scatter Plot",
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c"]  # xanh – cam – xanh lá
)

# giảm kích thước điểm
fig.update_traces(marker=dict(size=4))
fig.update_layout(width=1000, height=500)

fig.show()

**Nhận xét biểu đồ Scatter Plot R–F:**

Biểu đồ scatter R–F cho thấy ba cụm khách hàng được phân chia theo mô hình K-Means. Mặc dù các điểm có phần chồng lên nhau do dữ liệu lớn, có thể thấy xu hướng chung:

- Cluster 0 (xanh): Recency thấp, Frequency thấp → nhóm khách mới mua gần đây nhưng tần suất mua ít.

- Cluster 1 (cam): Recency thấp–trung bình, Frequency trung bình–cao → nhóm khách khá thường xuyên quay lại.

- Cluster 2 (xanh lá): Recency cao, Frequency thấp → khách đã lâu không quay lại và mua không nhiều.

### 3D Visualization

In [29]:
# Chuyển cluster thành dạng chuỗi để Plotly hiểu là phân loại
rfm['Cluster_str'] = rfm['Cluster'].astype(str)

fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster_str",  # dùng chuỗi để ra màu phân loại
    title="RFM Clustering – 3D View",
    opacity=0.7,
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c"]  # 3 màu rõ ràng
)

fig.update_traces(marker=dict(size=4))
fig.show()

**Nhận xét biểu đồ phân cụm 3D (R–F–M):**

Biểu đồ 3D cho thấy mô hình K-Means đã phân tách khách hàng thành 3 cụm khá rõ ràng:

- Cluster màu xanh dương: Recency cao, Frequency thấp, Monetary thấp → nhóm khách đã lâu không quay lại và chi tiêu ít.

- Cluster màu xanh lá: Recency trung bình, Frequency trung bình, Monetary trung bình → nhóm khách hoạt động mức vừa, mua không quá thường xuyên.

- Cluster màu cam: Recency thấp, Frequency cao, Monetary cao → nhóm khách hàng giá trị cao, mua thường xuyên và chi tiêu nhiều.

→ Việc quan sát cả ba chiều Recency–Frequency–Monetary giúp thấy rõ ràng cấu trúc phân cụm hơn so với biểu đồ 2D. Điều này cho thấy lựa chọn k = 3 là hợp lý cho bài toán phân nhóm khách hàng theo RFM.

## 7. Kết luận

Trong bài thực hành này, mô hình K-Means đã được áp dụng để phân nhóm khách hàng dựa trên bộ chỉ số RFM được trích xuất từ dữ liệu OnlineRetail. Sau quá trình làm sạch dữ liệu, loại bỏ outliers và chuẩn hóa, phương pháp Elbow cho thấy k = 3 là số cụm tối ưu.

Kết quả phân cụm cho thấy khách hàng có thể được chia thành 3 nhóm với đặc điểm khá rõ ràng:

- Cluster 0 (xanh dương): Khách hàng có Recency cao, Frequency thấp và chi tiêu thấp → nhóm đã lâu không quay lại, ít mua sắm.

- Cluster 1 (xanh lá): Khách có mức độ hoạt động trung bình, mua không quá thường xuyên nhưng vẫn duy trì tương tác.

- Cluster 2 (cam): Nhóm khách hàng giá trị cao, mua thường xuyên và chi tiêu lớn → đây là nhóm quan trọng cần được ưu tiên chăm sóc.

Việc phân nhóm như trên giúp doanh nghiệp hiểu rõ hành vi khách hàng và có thể đưa ra các chiến lược chăm sóc phù hợp cho từng nhóm, như giữ chân khách giá trị cao, kích hoạt khách kém tương tác hoặc thúc đẩy mua lại từ nhóm đã lâu không quay lại.

→ Mô hình K-Means trên RFM cho kết quả hợp lý và mang lại cái nhìn hữu ích cho hoạt động marketing và chăm sóc khách hàng.

# Kết thúc